In [ ]:
from __future__ import annotations

import jax
import jax.numpy as jnp
import equinox as eqx
import numpy as np
import optax
import pandas as pd
from diffrax import Dopri5, ODETerm, PIDController, SaveAt, diffeqsolve
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

from mxlpy import Model, Simulator
from mxlpy.jax import to_jax_export

# Universal Differential Equations for Plant Fluorescence

This notebook demonstrates a complete **UDE** (Universal Differential Equation) workflow
using mxlpy's JAX export.

**Scenario:**  A small mechanistic model captures the quenching dynamics of plant
chlorophyll fluorescence under a changing illumination protocol.  The *true* biological
system contains an unknown periodic forcing — simulating e.g. stomatal oscillations or
circadian rhythm crosstalk — that the mechanistic model does not account for.

A Neural Differential Equation (NDE) is appended as an additive correction to the
mechanistic RHS.  After training on trajectory data the NDE recovers the hidden periodic
term.

```
dQ/dt = k_ind·I − k_rel·Q         ← mechanistic  (JaxExport.rhs)
       + nn([t̄, Q̄, Ī])            ← unknown forcing  (Equinox MLP, to be learned)
```

## 1 · Mechanistic fluorescence model

The quenching state $Q$ builds up under illumination and relaxes spontaneously:

$$\frac{dQ}{dt} = k_\text{ind}\cdot I - k_\text{rel}\cdot Q$$

Fluorescence yield follows the Stern–Volmer relationship:

$$F(t) = \frac{1}{1 + Q(t)}$$

where $I$ is the photon flux density (PFD, μmol m⁻² s⁻¹) treated as a model parameter
that changes with each protocol phase.

In [ ]:
def linear_induction(k_ind: float, I: float) -> float:
    """Quenching induction proportional to light intensity."""
    return k_ind * I


def mass_action_relaxation(k_rel: float, Q: float) -> float:
    """First-order quenching relaxation."""
    return k_rel * Q


model = (
    Model()
    .add_variable("Q", 0.0)
    .add_parameters({"k_ind": 0.001, "k_rel": 0.1, "I": 100.0})
    .add_reaction(
        "induction",
        linear_induction,
        stoichiometry={"Q": 1},
        args=["k_ind", "I"],
    )
    .add_reaction(
        "relaxation",
        mass_action_relaxation,
        stoichiometry={"Q": -1},
        args=["k_rel", "Q"],
    )
)

print("Variables :", model.get_variable_names())
print("Parameters:", list(model.get_parameter_values()))

## 2 · Illumination protocol

Three phases of 100 s each, defined as an mxlpy `pd.DataFrame` protocol:

| Phase | Duration | PFD |
|-------|----------|-----|
| Low light   | 0 – 100 s   | 100 PFD  |
| High light  | 100 – 200 s | 2000 PFD |
| Medium light| 200 – 300 s | 500 PFD  |

Each row updates the parameter `I` for that segment.  The protocol is parsed into a list
of `(t_start, t_end, I_value)` segments used throughout the notebook — no piecewise
functions required.

In [ ]:
# mxlpy-native protocol: each row specifies parameter values valid until that time
protocol = pd.DataFrame(
    {"I": [100.0, 2000.0, 500.0]},
    index=pd.to_timedelta([100, 200, 300], unit="s"),
)

# Parse into (t_start, t_end, I_value) tuples for segment-wise integration
segments: list[tuple[float, float, float]] = []
t_cursor = 0.0
for td, row in protocol.iterrows():
    t_end = td.total_seconds()
    segments.append((t_cursor, t_end, float(row["I"])))
    t_cursor = t_end

T_TOTAL = float(segments[-1][1])
I_MAX = float(protocol["I"].max())

print("Protocol segments:")
for t0, t1, I_val in segments:
    print(f"  t=[{t0:.0f}, {t1:.0f}]  I={I_val:.0f} PFD")

# Step-function plot using protocol edges
t_edges = [0.0] + [t1 for _, t1, _ in segments]
I_vals = [I for _, _, I in segments]
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.stairs(I_vals, t_edges, color="goldenrod", linewidth=2.5)
ax.set(xlabel="Time (s)", ylabel="PFD (μmol m⁻² s⁻¹)", title="Illumination protocol")
ax.set_ylim(0, 2400)
plt.tight_layout()

## 3 · Export the mechanistic model to JAX

`to_jax_export` converts the mxlpy model to a differentiable RHS function via its
SymPy symbolic representation.  The returned `JaxExport` object is compatible with
Diffrax and supports `jax.jit`, `jax.grad`, and `jax.vmap`.

We capture a base parameter vector with the default `I = 100`.  Each protocol segment
will override this value by updating the relevant index — no piecewise time function
needed inside the RHS.

In [ ]:
export = to_jax_export(model)

print("variable_names :", export.variable_names)
print("parameter_names:", export.parameter_names)

# Index of the light intensity parameter in the parameter vector
I_IDX = export.parameter_names.index("I")
print(f"'I' is at parameter index {I_IDX}")

# Base parameter vector (I will be overridden per segment)
PARAMS_BASE = export.get_args()
Y0 = export.get_y0()

# Sanity check: exported RHS matches model() at t=0
dydt_jax = export.rhs(0.0, Y0, PARAMS_BASE)
dydt_model = model(0.0, list(Y0))
print(f"\nRHS at t=0, Q=0:  JAX={float(dydt_jax[0]):.5f}  model={dydt_model[0]:.5f}")

## 4 · Synthetic data — true model with hidden sinusoidal forcing

The *true* system includes a periodic term absent from the mechanistic model:

$$\frac{dQ}{dt} = k_\text{ind}\cdot I - k_\text{rel}\cdot Q
   + \underbrace{A\sin(\omega t)}_{\text{hidden forcing}}$$

We integrate this segment by segment — mirroring the protocol structure — using scipy
as a ground truth.  The NDE will need to recover the sinusoidal term.

In [ ]:
# Hidden parameters — unknown to the mechanistic model
A_TRUE = 0.3  # amplitude (quenching units)
OMEGA_TRUE = 2 * np.pi / 20.0  # angular frequency (period ≈ 20 s)

K_IND = model.get_parameter_values()["k_ind"]
K_REL = model.get_parameter_values()["k_rel"]

# Dense time grid matching the protocol range
t_data = np.linspace(0.0, T_TOTAL, 301)

# Integrate segment by segment — I is constant within each segment
t_parts: list[np.ndarray] = []
Q_parts: list[np.ndarray] = []
y_cur = [0.0]

for i, (t0_seg, t1_seg, I_val) in enumerate(segments):
    # First segment includes t=0; subsequent ones exclude the shared boundary
    if i == 0:
        t_pts = t_data[(t_data >= t0_seg) & (t_data <= t1_seg)]
    else:
        t_pts = t_data[(t_data > t0_seg) & (t_data <= t1_seg)]

    def true_rhs(t: float, y: np.ndarray, I: float = I_val) -> list[float]:
        return [K_IND * I - K_REL * y[0] + A_TRUE * np.sin(OMEGA_TRUE * t)]

    sol = solve_ivp(
        true_rhs,
        [t0_seg, t1_seg],
        y_cur,
        t_eval=t_pts,
        method="RK45",
        rtol=1e-7,
        atol=1e-9,
    )
    y_cur = [float(sol.y[0, -1])]
    t_parts.append(sol.t)
    Q_parts.append(sol.y[0])

t_data = np.concatenate(t_parts)  # 301 time points
Q_true = np.concatenate(Q_parts)
F_true = 1.0 / (1.0 + Q_true)

print(f"Data points : {len(t_data)}")
print(f"Q range     : [{Q_true.min():.2f}, {Q_true.max():.2f}]")
print(f"F range     : [{F_true.min():.3f}, {F_true.max():.3f}]")

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
ax1.stairs(I_vals, t_edges, color="goldenrod", lw=2)
ax1.set(ylabel="PFD", title="Illumination protocol")
ax2.plot(t_data, Q_true, color="steelblue", lw=1.5)
ax2.set(ylabel="Q", title="Quenching state (true, with hidden forcing)")
ax3.plot(t_data, F_true, color="forestgreen", lw=1.5)
ax3.set(xlabel="Time (s)", ylabel="F", title="Fluorescence yield (true)")
plt.tight_layout()

## 5 · Mechanistic reference simulation

`Simulator.simulate_protocol_time_course` integrates the model over all protocol phases,
updating the `I` parameter between segments automatically.  This gives the mechanistic
baseline — the residual between this and the true trajectory is what the NDE must learn.

In [ ]:
# simulate_protocol_time_course updates model.I for each segment
mech_sim = Simulator(model).simulate_protocol_time_course(protocol, t_data)
mech_df = pd.concat(mech_sim.variables)

Q_mech = mech_df["Q"].to_numpy()
F_mech = 1.0 / (1.0 + Q_mech)

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(t_data, Q_true, "k-", lw=1.5, label="True (with hidden forcing)")
axes[0].plot(t_data, Q_mech, "b--", lw=1.5, label="Mechanistic only")
axes[0].set(ylabel="Q", title="Quenching — true vs. mechanistic")
axes[0].legend()
axes[1].plot(t_data, F_true, "k-", lw=1.5, label="True")
axes[1].plot(t_data, F_mech, "b--", lw=1.5, label="Mechanistic only")
axes[1].set(xlabel="Time (s)", ylabel="F", title="Fluorescence — true vs. mechanistic")
axes[1].legend()
plt.tight_layout()

## 6 · Building the Universal Differential Equation

The UDE augments the mechanistic RHS with a small MLP:

$$\frac{dQ}{dt} = \underbrace{k_\text{ind}\cdot I - k_\text{rel}\cdot Q}_{\text{mechanistic}}
+\; \underbrace{\text{MLP}(\bar{t},\, \bar{Q},\, \bar{I})}_{\text{neural correction}}$$

Inputs to the MLP are normalised to $[0, 1]$:
$\bar{t} = t/T$, $\bar{Q} = Q/Q_\max$, $\bar{I} = I/I_\max$.

Because the light intensity $I$ is constant within each protocol segment, it is baked
directly into the parameter vector for that segment.  The `ude_rhs` reads $I$ from
`params[I_IDX]` — no time-dependent piecewise function is needed inside the ODE.

In [ ]:
# Normalisation constants
Q_MAX = float(Q_true.max()) * 1.5  # headroom above training range

# Pre-build one parameter vector per protocol segment (I baked in)
# and the corresponding time-point arrays for SaveAt
jax_segments: list[dict] = []
for i, (t0_seg, t1_seg, I_val) in enumerate(segments):
    # Mirror the t_data split used for true-data generation
    if i == 0:
        t_pts = t_data[(t_data >= t0_seg) & (t_data <= t1_seg)]
    else:
        t_pts = t_data[(t_data > t0_seg) & (t_data <= t1_seg)]
    jax_segments.append(
        {
            "t0": t0_seg,
            "t1": t1_seg,
            "params": PARAMS_BASE.at[I_IDX].set(I_val),  # JAX array with I overridden
            "t_eval": t_pts,  # NumPy array → concrete in JIT
        }
    )

print("Segment save-point counts:", [len(s["t_eval"]) for s in jax_segments])


def ude_rhs(t: float, y: jax.Array, args: tuple) -> jax.Array:
    """UDE RHS: mechanistic + additive neural correction.

    args = (params, nn_model)
    I is fixed for the current segment and already in params[I_IDX].
    """
    params, nn_model = args
    mech = export.rhs(t, y, params)
    I_norm = params[I_IDX] / I_MAX
    nn_in = jnp.array([t / T_TOTAL, y[0] / Q_MAX, I_norm])
    return mech + nn_model(nn_in)


# Small MLP: 3 → 32 → 32 → 1 with tanh activations
key = jax.random.PRNGKey(0)
nn = eqx.nn.MLP(3, 1, width_size=32, depth=2, key=key, activation=jnp.tanh)

n_params = sum(p.size for p in jax.tree_util.tree_leaves(eqx.filter(nn, eqx.is_array)))
print(f"NDE parameters: {n_params}")

## 7 · Training

We minimise the mean-squared error between the UDE's predicted $Q$ trajectory and the
synthetic training data.  The UDE is integrated **segment by segment**, matching the
protocol: each segment uses its fixed $I$ value and chains the final state to the next
segment's initial condition.

Gradients flow through all three Diffrax solves via `eqx.filter_value_and_grad`.

In [ ]:
Q_TARGET = jnp.array(Q_true)  # shape (301,)


def simulate_ude(nn_model: eqx.nn.MLP) -> jax.Array:
    """Integrate the UDE over all protocol segments, chaining end-states."""
    y = export.get_y0()
    all_ys: list[jax.Array] = []

    for seg in jax_segments:
        sol = diffeqsolve(
            ODETerm(ude_rhs),
            Dopri5(),
            t0=seg["t0"],
            t1=seg["t1"],
            dt0=0.5,
            y0=y,
            saveat=SaveAt(ts=seg["t_eval"]),
            stepsize_controller=PIDController(rtol=1e-3, atol=1e-5),
            args=(seg["params"], nn_model),
            max_steps=4096,
        )
        y = sol.ys[-1]  # chain state to next segment
        all_ys.append(sol.ys)

    return jnp.concatenate(all_ys, axis=0)  # (301, 1)


def compute_loss(nn_model: eqx.nn.MLP) -> jax.Array:
    """MSE between UDE prediction and training trajectory."""
    Q_pred = simulate_ude(nn_model)
    return jnp.mean((Q_pred[:, 0] - Q_TARGET) ** 2)


@eqx.filter_jit
def make_step(
    nn_model: eqx.nn.MLP,
    opt_state: optax.OptState,
) -> tuple[eqx.nn.MLP, optax.OptState, jax.Array]:
    """Single gradient-descent step."""
    loss, grads = eqx.filter_value_and_grad(compute_loss)(nn_model)
    updates, new_opt_state = optimizer.update(grads, opt_state, nn_model)
    new_nn = eqx.apply_updates(nn_model, updates)
    return new_nn, new_opt_state, loss


optimizer = optax.adam(3e-3)
opt_state = optimizer.init(eqx.filter(nn, eqx.is_array))

print("Compiling ... (first step takes longer due to JIT)")
losses: list[float] = []
for epoch in range(400):
    nn, opt_state, loss = make_step(nn, opt_state)
    losses.append(float(loss))
    if epoch % 50 == 0:
        print(f"  epoch {epoch:3d}  loss = {float(loss):.5f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(losses, color="royalblue", lw=1.5)
ax.set(xlabel="Epoch", ylabel="MSE loss (log scale)", title="UDE training loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()

## 8 · Results

### 8.1 Trajectory fit

In [ ]:
Q_ude = np.array(simulate_ude(nn))[:, 0]
F_ude = 1.0 / (1.0 + Q_ude)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(t_data, Q_true, "k-", lw=2, label="True (hidden forcing)")
axes[0].plot(t_data, Q_ude, "r--", lw=1.8, label="UDE (trained)")
axes[0].plot(t_data, Q_mech, "b:", lw=1.5, label="Mechanistic only")
axes[0].set(ylabel="Quenching state Q")
axes[0].legend()

axes[1].plot(t_data, F_true, "k-", lw=2, label="True")
axes[1].plot(t_data, F_ude, "r--", lw=1.8, label="UDE (trained)")
axes[1].plot(t_data, F_mech, "b:", lw=1.5, label="Mechanistic only")
axes[1].set(xlabel="Time (s)", ylabel="Fluorescence F")
axes[1].legend()

plt.suptitle("UDE vs. mechanistic model vs. true trajectory", y=1.01)
plt.tight_layout()

### 8.2 What did the NDE learn?

We evaluate the trained NDE along the true $Q$ trajectory and compare its output
to the hidden sinusoidal forcing $A\sin(\omega t)$.  The light intensity $I$ for each
time point is read from the protocol segments — no piecewise function needed.

In [ ]:
# Build I array for every time point directly from protocol segments
I_arr = np.zeros(len(t_data))
for i, (t0_seg, t1_seg, I_val) in enumerate(segments):
    mask = (
        (t_data >= t0_seg) & (t_data <= t1_seg)
        if i == 0
        else (t_data > t0_seg) & (t_data <= t1_seg)
    )
    I_arr[mask] = I_val

# Vectorised NDE evaluation
nn_inputs = jnp.array(
    np.stack([t_data / T_TOTAL, Q_true / Q_MAX, I_arr / I_MAX], axis=1)
)
nde_output = np.array(jax.vmap(nn)(nn_inputs)[:, 0])
true_forcing = A_TRUE * np.sin(OMEGA_TRUE * t_data)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t_data, true_forcing, "k-", lw=2, label=f"True forcing: {A_TRUE}·sin(ωt)")
ax.plot(t_data, nde_output, "r--", lw=1.8, label="NDE output (learned)")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set(
    xlabel="Time (s)",
    ylabel="dQ/dt contribution",
    title="Recovered sinusoidal forcing vs. true hidden term",
)
ax.legend()
plt.tight_layout()

### 8.3 Residual analysis

In [ ]:
residual_ude = Q_ude - Q_true
residual_mech = Q_mech - Q_true

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_data, residual_mech, "b:", lw=1.5, alpha=0.8, label="Mechanistic only")
ax.plot(t_data, residual_ude, "r--", lw=1.5, alpha=0.8, label="UDE (trained)")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="Time (s)", ylabel="Q_pred − Q_true", title="Prediction residuals")
ax.legend()
plt.tight_layout()

print(f"RMSE mechanistic only : {np.sqrt(np.mean(residual_mech**2)):.4f}")
print(f"RMSE UDE (trained)    : {np.sqrt(np.mean(residual_ude**2)):.4f}")